# Brique 2.2 — Exercices moyens

Un cran plus loin : plusieurs properties qui interagissent, protéger une liste interne (pas seulement un
nombre), et un choix de conception (lever une erreur ou corriger automatiquement).

### Exercice 1 — Rectangle avec dimensions validées et grandeurs calculées
Écris une classe `Rectangle` dont `__init__` reçoit `largeur` et `hauteur`, chacune étant une `@property` dont
le setter lève `ValueError` si la valeur est inférieure ou égale à 0. Ajoute deux properties en lecture seule
(sans setter) : `aire` et `perimetre`, calculées à partir de `self.largeur` et `self.hauteur`.

In [ ]:
class Rectangle:
    def __init__(self, largeur, hauteur):
        raise NotImplementedError

    @property
    def largeur(self):
        raise NotImplementedError

    @largeur.setter
    def largeur(self, valeur):
        raise NotImplementedError

    @property
    def hauteur(self):
        raise NotImplementedError

    @hauteur.setter
    def hauteur(self, valeur):
        raise NotImplementedError

    @property
    def aire(self):
        raise NotImplementedError

    @property
    def perimetre(self):
        raise NotImplementedError

In [ ]:
r = Rectangle(3, 5)
assert r.aire == 15
assert r.perimetre == 16

r.largeur = 10
assert r.aire == 50   # l'aire doit se recalculer automatiquement, sans rien faire de plus

for valeur_invalide in (0, -5):
    try:
        r.hauteur = valeur_invalide
        assert False, "aurait du lever ValueError"
    except ValueError:
        pass
print("OK")

### Exercice 2 — Protéger une LISTE interne, pas seulement un nombre
Écris une classe `CompteBancaire` dont `__init__` reçoit `solde_initial`, stocke `_solde` et initialise
`_historique = []`. Ajoute `deposer(self, montant)` et `retirer(self, montant)` qui modifient `_solde` et
ajoutent une ligne à `_historique` (par exemple `f"depot de {montant}"` ou `f"retrait de {montant}"`). Ajoute une
`@property historique` en lecture seule.

**Le piège à éviter** : si `historique` renvoie directement `self._historique` (la vraie liste interne), alors
quelqu'un pourrait faire `compte.historique.append("faux")` et modifier l'historique réel de l'intérieur, sans
passer par aucune méthode — exactement le problème qu'on cherche à éviter depuis le début de ce notebook, mais
appliqué à une liste plutôt qu'à un nombre. La property doit donc renvoyer une **copie** de la liste
(`list(self._historique)` ou `self._historique.copy()`), pas la liste elle-même.

*Indice sur `solde` : réutilise le même principe que dans `00_cours.ipynb` (property + setter qui refuse les
valeurs négatives).*

In [ ]:
class CompteBancaire:
    def __init__(self, solde_initial):
        raise NotImplementedError

    @property
    def solde(self):
        raise NotImplementedError

    @solde.setter
    def solde(self, valeur):
        raise NotImplementedError

    def deposer(self, montant):
        raise NotImplementedError

    def retirer(self, montant):
        raise NotImplementedError

    @property
    def historique(self):
        raise NotImplementedError

In [ ]:
compte = CompteBancaire(100)
compte.deposer(50)
compte.retirer(30)

assert compte.solde == 120
assert len(compte.historique) == 2

# Le piège : modifier la liste renvoyée ne doit RIEN changer en interne
copie = compte.historique
copie.append("ligne frauduleuse")
assert len(compte.historique) == 2, "la vraie liste interne a été modifiée de l'extérieur : il manque une copie"

try:
    compte.retirer(10000)
    assert False, "aurait du lever ValueError"
except ValueError:
    pass
print("OK")

### Exercice 3 — Un choix de conception différent : corriger plutôt que refuser
Jusqu'ici, chaque setter invalide **levait une exception**. Ce n'est pas la seule approche valable : parfois, on
préfère **corriger automatiquement** une valeur hors limites plutôt que refuser. Écris une classe `Jauge`
(pense à une jauge de batterie), dont `__init__` reçoit `niveau`, une `@property` dont le setter **ramène
automatiquement** toute valeur en dehors de `[0, 100]` à la borne la plus proche (pas de `ValueError` ici) :
une valeur de `150` devient `100`, une valeur de `-20` devient `0`.

*Indice : `max(0, min(valeur, 100))` ramène `valeur` dans l'intervalle `[0, 100]` en une seule expression.*

In [ ]:
class Jauge:
    def __init__(self, niveau):
        raise NotImplementedError

    @property
    def niveau(self):
        raise NotImplementedError

    @niveau.setter
    def niveau(self, valeur):
        raise NotImplementedError

In [ ]:
j = Jauge(50)
assert j.niveau == 50

j.niveau = 150
assert j.niveau == 100   # pas d'erreur levée : la valeur est simplement ramenée à la borne

j.niveau = -20
assert j.niveau == 0

j2 = Jauge(500)   # la correction doit s'appliquer dès la création aussi
assert j2.niveau == 100
print("OK")

**Réfléchis une seconde avant de continuer** : dans l'exercice 2 (le solde du compte), on a choisi de lever une
erreur plutôt que de corriger automatiquement (par exemple en refusant le retrait plutôt qu'en vidant le compte à
0). Pourquoi ce choix est-il le bon pour un solde bancaire, alors que "corriger automatiquement" est le bon choix
pour une jauge de batterie ? *(Pas de test à faire ici, juste une question à te poser — la réponse : une jauge
qui dépasse 100 n'est pas une erreur de l'appelant, c'est une situation normale et sans conséquence à corriger
silencieusement ; un retrait bancaire trop élevé, lui, est une opération invalide que l'appelant doit savoir
qu'elle a échoué, sinon de l'argent "disparaîtrait" silencieusement.)*

---
**Brique 2.2 terminée.** Coche `02_encapsulation` dans [ROADMAP.md](../../ROADMAP.md). Prochaine étape :
`03_composition`.